# ray_tpu smoke test

Light-touch import + API-surface check for the `ray_tpu` package (Ray helper for TPU pods).
Most APIs require a live TPU pod, so we exercise only the parts that work without TPU hardware.

ray_tpu imports ray but does not declare it as a dependency: `pip install ray_tpu ray`.

In [ ]:
# Example 1: inspect top-level public API surface.
import ray_tpu

public_names = [n for n in dir(ray_tpu) if not n.startswith('_')]
print(f"ray_tpu exposes {len(public_names)} public name(s):")
for name in public_names:
    obj = getattr(ray_tpu, name)
    print(f"  {name}: {type(obj).__name__}")

assert len(public_names) > 0, 'ray_tpu has no public names'

In [ ]:
# Example 2: verify expected callable / class symbols exist on the module.
import inspect

import ray_tpu

callables = [n for n in dir(ray_tpu) if not n.startswith('_') and callable(getattr(ray_tpu, n))]
print(f"{len(callables)} callable(s) on ray_tpu:")
for name in callables:
    obj = getattr(ray_tpu, name)
    kind = 'class' if inspect.isclass(obj) else 'function'
    print(f"  {name} ({kind})")

assert callables, 'expected at least one callable on ray_tpu'

In [ ]:
# Example 3: introspect signatures of the public callables (smoke-tests they're importable + bindable).
import inspect

import ray_tpu

inspected = 0
for name in dir(ray_tpu):
    if name.startswith('_'):
        continue
    obj = getattr(ray_tpu, name)
    if not callable(obj):
        continue
    try:
        sig = inspect.signature(obj)
        print(f"{name}{sig}")
        inspected += 1
    except (TypeError, ValueError) as e:
        print(f"{name}: signature unavailable ({type(e).__name__})")

print(f"\ninspected {inspected} signature(s)")

In [ ]:
# Example 4: confirm ray (imported by ray_tpu but not declared as a dependency) is importable and reachable from ray_tpu.
import ray
import ray_tpu

print(f"ray.__version__ = {ray.__version__}")
print(f"ray_tpu module file: {ray_tpu.__file__ if hasattr(ray_tpu, '__file__') else ray_tpu.__path__}")

# The package source references ray in its impl; we don't actually init ray here
# (no TPU available + ray.init() is heavyweight). We just confirm the dep chain resolves.
assert ray.__version__, 'ray must report a version'

In [ ]:
# Example 5: re-import + verify module identity is stable (module cache works, no side-effect surprises).
# ray_tpu is a flat single-file module, so identity is anchored on __file__ (no __path__).
import sys

import ray_tpu

first_id = id(ray_tpu)
first_file = ray_tpu.__file__

# Re-import via sys.modules — should be the same object.
import ray_tpu as ray_tpu_again
second_id = id(ray_tpu_again)

assert first_id == second_id, 'ray_tpu module identity changed across imports'
assert 'ray_tpu' in sys.modules, 'ray_tpu missing from sys.modules'
assert first_file == ray_tpu_again.__file__, '__file__ changed across imports'
print(f"ray_tpu import is stable; __file__ = {first_file}")